# 2. Data Clean Up

In [1]:
import pandas as pd
import time
import ssl
from fredapi import Fred

gold_data = pd.read_csv("Gold-Silver-GeopoliticalRisk_HistoricalData.csv")
gold_data['DATE'] = pd.to_datetime(gold_data['DATE'])
gold_data.set_index('DATE', inplace=True)
gold_data = gold_data[gold_data.index >= '1999-01-01']
gold_data

# bypass SSL verification if encountering certificate verification errors
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# initialize FRED API
fred = Fred(api_key="77607617679b9e388bc1c17f17bf0874")

start = '1991-01-01'
end = '2025-09-10'

# map column names to FRED Series IDs
series_map = {
    'interest_rate': 'REAINTRATREARAT10Y',
    'inflation_rate': 'FPCPITOTLZGUSA',
    'expected_inflation': 'EXPINF10YR',
    'gdp': 'GDP',
    'us_m2_money_stock': 'WM2NS',
    'wti_crude_oil_price': 'POILWTIUSDM'
}

# fetch data sequentially with a delay to prevent HTTP 500 Errors
data = {}
for column_name, series_id in series_map.items():
    print(f"Fetching {column_name}...")
    try:
        data[column_name] = fred.get_series(series_id, observation_start=start, observation_end=end)

        # 1-second pause to avoid overloading the API
        time.sleep(1)
    except Exception as e:
        print(f"  -> Error fetching {column_name}: {e}")

# combine into a single DataFrame
df1 = pd.DataFrame(data)
print("Data fetching complete!")

Fetching interest_rate...
Fetching inflation_rate...
Fetching expected_inflation...
Fetching gdp...
Fetching us_m2_money_stock...
Fetching wti_crude_oil_price...
Data fetching complete!


In [2]:
import yfinance as yf

# gold_futures = yf.download('MGC=F', start=start, end=end)
vix = yf.download('^VIX', start=start, end=end)
us_dollar = yf.download('DX-Y.NYB', start=start, end=end)
s_and_p_500 = yf.download('^GSPC', start=start, end=end)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [3]:
raw_data = pd.concat([df1, s_and_p_500, vix, us_dollar, gold_data], axis=1)
raw_data.isna().sum()
raw_data.head()

C:\Users\Jasper\AppData\Local\Temp\ipykernel_93544\930144153.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  raw_data = pd.concat([df1, s_and_p_500, vix, us_dollar, gold_data], axis=1)


,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"(Close, ^GSPC)","(High, ^GSPC)","(Low, ^GSPC)","(Open, ^GSPC)",...,GOLD_CHANGE_%,SILVER_PRICE,SILVER_OPEN,SILVER_HIGH,SILVER_LOW,SILVER_CHANGE_%,GPRD,GPRD_ACT,GPRD_THREAT,EVENT
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,330.750000,326.450012,330.200012,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,326.529999,321.899994,326.459991,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,322.350006,318.869995,321.910004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,320.970001,315.440002,320.970001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



## Model Setup with Target Variable and Redundant Variable Deletion

### Model 1: Gold Price Model
- **Target variable:** `GOLD_PRICE`. Other gold columns are removed because they describe the same asset directly, and silver columns are also removed so the model focuses on external drivers rather than another closely related metal price.
- **Market assets kept:** only **Close** is kept for the **S&P 500** (broad equity conditions), **VIX** (market stress), and **U.S. dollar index** (dollar strength). These give one interpretable price signal per asset and avoid repeated OHLC information.
- **Geopolitical variables kept/removed:** keep **`GPRD`** as the main geopolitical risk measure. Remove **`GPRD_ACT`** and **`GPRD_THREAT`** for the first model because they are closely related sub-indexes and may add overlapping information. Remove **`EVENT`** because it is sparse and mostly missing.



### Model 2: Silver Catch-Up Model
- **Target variable:** `Future Silver Price (after 1 quarter) with new gold_silver_ratio(Gold/silver) as one of the predictors. This directly matches the idea that silver may catch up when the gold/silver ratio becomes high.
- **Market assets kept:** only **Close** is kept for the **S&P 500**, **VIX**, and **U.S. dollar index**, along with the macro variables. Keep **`GPRD`** as the main geopolitical control.
- **Columns removed:** repeated **Open/High/Low/Volume** fields are removed for each market asset, and raw current gold and silver columns are removed for now as they may be utilized later. **`GPRD_ACT`**, **`GPRD_THREAT`**, and **`EVENT`** are also removed for same reasons as model 1.


In [4]:
# Model 1: only keep GOLD_CLOSE out of gold variables, remove silver variables, then drop repeated Open/High/low, extra GPR variables.
df = raw_data.copy()
df.columns = df.columns.map(str)

model1_df = df.drop(columns=[
    "('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')","('Volume', '^GSPC')",
    "('High', '^VIX')","('Low', '^VIX')","('Open', '^VIX')","('Volume', '^VIX')",
    "('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",
    "GOLD_OPEN","GOLD_HIGH","GOLD_LOW","GOLD_CHANGE_%",
    "SILVER_PRICE","SILVER_OPEN","SILVER_HIGH","SILVER_LOW","SILVER_CHANGE_%",
    "GPRD_ACT","GPRD_THREAT","EVENT"
], errors="ignore")

model1_df.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('Close', '^VIX')","('Close', 'DX-Y.NYB')",GOLD_PRICE,GPRD
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,83.070000,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,26.620001,82.790001,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,27.930000,82.809998,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,27.190001,83.360001,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,28.950001,84.669998,NaN,NaN


In [5]:
# Model 2: future silver price target, ratio as predictor, then drop repeated open/high/low, raw gold, extra GPR variables.

df = raw_data.copy()
df.columns = df.columns.map(str)  # string cols
df["gold_silver_ratio"] = df["GOLD_PRICE"] / df["SILVER_PRICE"]  # predictor
df["future_silver_price_qtr"] = df["SILVER_PRICE"].shift(-63)  # ~1 trading quarter ahead

model2_df = df.drop(columns=[
    "(High, ^GSPC)","(Low, ^GSPC)","(Open, ^GSPC)","(Volume, ^GSPC)",
    "(High, ^VIX)","(Low, ^VIX)","(Open, ^VIX)","(Volume, ^VIX)",
    "(High, DX-Y.NYB)","(Low, DX-Y.NYB)","(Open, DX-Y.NYB)","(Volume, DX-Y.NYB)",
    "GOLD_OPEN","GOLD_HIGH","GOLD_LOW","GOLD_CHANGE_%", "GOLD_PRICE",	"SILVER_PRICE",
    "SILVER_OPEN","SILVER_HIGH","SILVER_LOW","SILVER_CHANGE_%",
    "GPRD_ACT","GPRD_THREAT","EVENT"
], errors="ignore")
model2_df.head()


,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')",...,"('Open', '^VIX')","('Volume', '^VIX')","('Close', 'DX-Y.NYB')","('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",GPRD,gold_silver_ratio,future_silver_price_qtr
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.070000,83.430000,83.010002,83.370003,0.0,NaN,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,330.750000,326.450012,330.200012,...,26.620001,0.0,82.790001,83.080002,82.660004,82.930000,0.0,NaN,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,326.529999,321.899994,326.459991,...,27.930000,0.0,82.809998,82.930000,82.680000,82.889999,0.0,NaN,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,322.350006,318.869995,321.910004,...,27.190001,0.0,83.360001,83.519997,82.599998,82.669998,0.0,NaN,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,320.970001,315.440002,320.970001,...,28.950001,0.0,84.669998,85.110001,84.540001,84.870003,0.0,NaN,NaN,NaN


### Removing NAs

In [6]:
model1_df = model1_df.dropna(subset=["GOLD_PRICE"])  # drop rows with no gold target
x1_cols = model1_df.columns.drop("GOLD_PRICE")  # non-target cols
model1_df[x1_cols] = model1_df[x1_cols].ffill()  # carry last value forward in all feature variables if NA
model1_df = model1_df.dropna()  # drop the few leading rows still NA
model1_df.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('Close', '^VIX')","('Close', 'DX-Y.NYB')",GOLD_PRICE,GPRD
2008-01-01,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1468.359985,22.500000,76.699997,833.7,63.37
2008-01-02,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1447.160034,23.170000,75.970001,857.2,95.00
2008-01-03,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1447.160034,22.490000,75.889999,863.4,82.81
2008-01-04,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1411.630005,23.940001,75.790001,861.5,87.35
2008-01-07,1.399491,3.8391,2.174666,14706.538,7551.9,92.982609,1416.180054,23.790001,76.169998,857.9,78.18


In [7]:
x2_cols = model2_df.columns.drop("future_silver_price_qtr")  # features only
model2_df[x2_cols] = model2_df[x2_cols].ffill()  # carry last value forward
model2_df = model2_df.dropna()  # remove remaining NAs
model2_df.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')",...,"('Open', '^VIX')","('Volume', '^VIX')","('Close', 'DX-Y.NYB')","('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",GPRD,gold_silver_ratio,future_silver_price_qtr
1999-01-04,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1228.099976,1248.810059,1219.099976,1229.229980,...,25.379999,0.0,93.440002,94.150002,93.050003,93.889999,0.0,54.95,58.168016,5.04
1999-01-05,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1244.780029,1246.109985,1228.099976,1228.099976,...,25.920000,0.0,93.470001,93.820000,93.080002,93.440002,0.0,60.68,56.934524,5.16
1999-01-06,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1272.339966,1272.500000,1244.780029,1244.780029,...,23.360001,0.0,94.529999,94.849998,93.209999,93.739998,0.0,62.19,55.657640,4.93
1999-01-07,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1269.729980,1272.339966,1257.680054,1272.339966,...,24.420000,0.0,93.720001,94.480003,93.449997,94.419998,0.0,68.03,55.533333,4.97
1999-01-08,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1275.089966,1278.239990,1261.819946,1269.729980,...,22.950001,0.0,94.349998,94.820000,93.660004,93.800003,0.0,70.81,55.056711,4.97


### Variable Description
This table lists the main working variables used across both models covering **1991-01-01 to 2025-09-09**.

| Variable (units) | What it measures |
|---|---|
| `interest_rate` (%) | U.S. 10-year real interest rate. |
| `inflation_rate` (annual %) | U.S. CPI inflation rate. |
| `expected_inflation` (%) | U.S. expected average inflation over the next 10 years. |
| `gdp` (billions of USD, SAAR) | U.S. nominal gross domestic product. |
| `us_m2_money_stock` (billions of USD) | U.S. M2 money supply / broad household liquidity. |
| `wti_crude_oil_price` (USD/barrel) | Global WTI crude oil price. |
| `(Close, ^GSPC)` (index level) | S&P 500 closing level, measuring broad U.S. stock market performance. |
| `(Close, ^VIX)` (index level) | VIX closing level, measuring expected near-term U.S. stock market volatility. |
| `(Close, DX-Y.NYB)` (index level) | U.S. Dollar Index closing level, measuring U.S. dollar strength against a basket of major currencies. |
| `GPRD` (index, 1985–2019 = 100) | Daily geopolitical risk index. |
| `GOLD_PRICE` (USD-based gold price series) | Daily gold price level. |
| `SILVER_PRICE` (USD-based silver price series) | Daily silver price level. |
| `gold_silver_ratio` (unitless ratio) | Gold price divided by silver price. |
| `future_silver_price_qtr` (same units as `SILVER_PRICE`) | Silver price about one trading quarter later. |